In [1]:
from pathlib import Path
import os

cwd = Path.cwd()
if cwd.name == "notebooks" and (cwd.parent / "data").exists():
    os.chdir(cwd.parent)

print("CWD set to:", os.getcwd())

CWD set to: c:\Users\Borna\Desktop\w2v-hr


In [3]:
from __future__ import annotations

from pathlib import Path
import os
import random
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
print("CWD:", os.getcwd())
print("ROOT:", ROOT)

CORPUS_PATH = ROOT / "data" / "work" / "hrwac_work.txt"
CLEAN_PATH  = ROOT / "data" / "work" / "hrwac_work_clean.txt"

ANALOGIES_DIR  = ROOT / "analogies" 
SEEDS_DIR      = ANALOGIES_DIR / "seeds"
ANALOGIES_PATH = ANALOGIES_DIR / "hr_analogies.txt"
MODELS_DIR     = ROOT / "models"
RESULTS_DIR    = ROOT / "results"

print("CORPUS_PATH:", CORPUS_PATH)
print("exists:", CORPUS_PATH.exists())
assert CORPUS_PATH.exists(), f"Missing: {CORPUS_PATH}"

# 3) Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Setup OK.")
print("Raw corpus:", CORPUS_PATH)


MAKE_CLEAN_CORPUS = True  

def keep_token(tok: str) -> bool:
    if len(tok) < 2:
        return False
    if tok.isdigit():
        return False
    if any(c.isalpha() for c in tok) and any(c.isdigit() for c in tok):
        return False
    return True

if MAKE_CLEAN_CORPUS:
    if not CLEAN_PATH.exists():
        kept_sents = 0
        with CORPUS_PATH.open("r", encoding="utf-8") as fin, CLEAN_PATH.open("w", encoding="utf-8") as fout:
            for line in fin:
                toks = [t for t in line.strip().split() if keep_token(t)]
                if toks:
                    fout.write(" ".join(toks) + "\n")
                    kept_sents += 1
        print("Created clean corpus:", CLEAN_PATH, "| sentences:", f"{kept_sents:,}")
    else:
        print("Clean corpus already exists:", CLEAN_PATH)

TRAIN_CORPUS = CLEAN_PATH if (MAKE_CLEAN_CORPUS and CLEAN_PATH.exists()) else CORPUS_PATH
print("TRAIN_CORPUS:", TRAIN_CORPUS)

CWD: c:\Users\Borna\Desktop\w2v-hr
ROOT: c:\Users\Borna\Desktop\w2v-hr
CORPUS_PATH: c:\Users\Borna\Desktop\w2v-hr\data\work\hrwac_work.txt
exists: True
Setup OK.
Raw corpus: c:\Users\Borna\Desktop\w2v-hr\data\work\hrwac_work.txt
Clean corpus already exists: c:\Users\Borna\Desktop\w2v-hr\data\work\hrwac_work_clean.txt
TRAIN_CORPUS: c:\Users\Borna\Desktop\w2v-hr\data\work\hrwac_work_clean.txt


In [4]:
from pathlib import Path

total_tokens = 0
with CORPUS_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        total_tokens += len(line.strip().split())

print("Ukupan broj tokena:", total_tokens)

Ukupan broj tokena: 79559758


In [3]:
for i, line in enumerate(CORPUS_PATH.open("r", encoding="utf-8")):
    print(line.strip()[:200])
    if i >= 2:
        break

adam se otpočetka nudi tri benzinska motora 12 70 ks 14 87 ks 14 100 ks
svi motori dolaze kombinaciji ručnim mjenjačem pet stupnjeva prijenosa
dizelski motor nije uopće planu jer opel ističe kako sva tri motora kombiniranom načinu vožnje troše oko litara mogu se kupiti ili bez start stop sustava


In [ ]:
from __future__ import annotations
from pathlib import Path
from collections import Counter
import math

def percentile(sorted_vals, p: float):
    if not sorted_vals:
        return None
    k = (len(sorted_vals) - 1) * p
    f = math.floor(k)
    c = math.ceil(k)
    if f == c:
        return sorted_vals[int(k)]
    return sorted_vals[f] * (c - k) + sorted_vals[c] * (k - f)

def is_punctish(tok: str) -> bool:
    return all(not ch.isalnum() and ch not in "čćđšžČĆĐŠŽ" for ch in tok)

def is_alnum(tok: str) -> bool:
    return any(c.isalpha() for c in tok) and any(c.isdigit() for c in tok)

def deep_eda(path: Path, max_sents: int | None = None):
    freq = Counter()
    sent_lens = []

    total_sents = 0
    total_tokens = 0

    numeric = alnum = punctish = short = 0

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            toks = line.strip().split()
            if not toks:
                continue
            total_sents += 1
            if max_sents is not None and total_sents > max_sents:
                break

            L = len(toks)
            sent_lens.append(L)
            total_tokens += L
            freq.update(toks)

            for t in toks:
                if len(t) < 2:
                    short += 1
                if t.isdigit():
                    numeric += 1
                if is_alnum(t):
                    alnum += 1
                if is_punctish(t):
                    punctish += 1

    sent_lens.sort()

    hapax = sum(1 for _, c in freq.items() if c == 1)

    total = total_tokens
    topN = [100, 1000, 10000]
    coverage = {}
    running = 0
    for i, (_, c) in enumerate(freq.most_common(max(topN))):
        running += c
        if (i + 1) in topN:
            coverage[i + 1] = running / total

    return {
        "sentences": total_sents,
        "tokens": total_tokens,
        "vocab": len(freq),
        "hapax": hapax,
        "hapax_ratio": hapax / len(freq) if freq else 0,
        "mean_len": total_tokens / total_sents if total_sents else 0,
        "median_len": percentile(sent_lens, 0.5),
        "p95_len": percentile(sent_lens, 0.95),
        "p99_len": percentile(sent_lens, 0.99),
        "token_ratios": {
            "numeric": numeric / total if total else 0,
            "alnum": alnum / total if total else 0,
            "punctish": punctish / total if total else 0,
            "short(<2)": short / total if total else 0,
        },
        "coverage_topN": coverage,
        "top_30": freq.most_common(30),
    }

deep = deep_eda(TRAIN_CORPUS, max_sents=500_000)  
deep

{'sentences': 500001,
 'tokens': 8389659,
 'vocab': 414123,
 'hapax': 209436,
 'hapax_ratio': 0.5057338037249802,
 'mean_len': 16.779284441431116,
 'median_len': 14.0,
 'p95_len': 39.0,
 'p99_len': 62.0,
 'token_ratios': {'numeric': 0.0,
  'alnum': 0.0,
  'punctish': 0.0,
  'short(<2)': 0.0},
 'coverage_topN': {100: 0.28011639090456475,
  1000: 0.47651567244866566,
  10000: 0.737495290333016},
 'top_50': [('je', 270740),
  ('se', 161719),
  ('na', 152540),
  ('da', 123777),
  ('za', 112368),
  ('su', 90344),
  ('od', 68580),
  ('ne', 52674),
  ('koji', 51436),
  ('to', 43853),
  ('će', 43047),
  ('što', 41743),
  ('te', 36792),
  ('iz', 36171),
  ('kao', 33831),
  ('ili', 31010),
  ('kako', 30937),
  ('bi', 30623),
  ('do', 26491),
  ('nije', 26150),
  ('koje', 25954),
  ('sve', 24608),
  ('ali', 24239),
  ('sa', 23789),
  ('sam', 23619),
  ('godine', 21042),
  ('koja', 20219),
  ('samo', 19039),
  ('po', 18581),
  ('biti', 18503),
  ('više', 17463),
  ('jer', 17346),
  ('pa', 16885),


In [5]:
print("=== Deep EDA summary ===")
print("Sentences:", f"{deep['sentences']:,}")
print("Tokens:", f"{deep['tokens']:,}")
print("Vocab:", f"{deep['vocab']:,}")
print("Hapax:", f"{deep['hapax']:,}", f"({deep['hapax_ratio']:.2%} of vocab)")
print()
print("Sentence length: mean", round(deep["mean_len"],2), "median", deep["median_len"], "p95", deep["p95_len"], "p99", deep["p99_len"])
print()
print("Token ratios:")
for k,v in deep["token_ratios"].items():
    print(f"  {k:10s} {v:.3%}")
print()
print("Coverage by top-N tokens:")
for n, cov in deep["coverage_topN"].items():
    print(f"  top {n:5d}: {cov:.2%}")
print()
print("Top-20 tokens:")
for w,c in deep["top_50"][:20]:
    print(f"  {w:15s} {c:,}")

=== Deep EDA summary ===
Sentences: 500,001
Tokens: 8,389,659
Vocab: 414,123
Hapax: 209,436 (50.57% of vocab)

Sentence length: mean 16.78 median 14.0 p95 39.0 p99 62.0

Token ratios:
  numeric    0.000%
  alnum      0.000%
  punctish   0.000%
  short(<2)  0.000%

Coverage by top-N tokens:
  top   100: 28.01%
  top  1000: 47.65%
  top 10000: 73.75%

Top-20 tokens:
  je              270,740
  se              161,719
  na              152,540
  da              123,777
  za              112,368
  su              90,344
  od              68,580
  ne              52,674
  koji            51,436
  to              43,853
  će              43,047
  što             41,743
  te              36,792
  iz              36,171
  kao             33,831
  ili             31,010
  kako            30,937
  bi              30,623
  do              26,491
  nije            26,150


In [6]:
vocab_probe = deep_eda(TRAIN_CORPUS, max_sents=2_000_000)["freq"]
total = sum(vocab_probe.values())

for t in [20, 50, 100, 200]:
    vs = sum(1 for _,c in vocab_probe.items() if c >= t)
    cov = sum(c for _,c in vocab_probe.items() if c >= t) / total
    print(t, "vocab", vs, "token_coverage", f"{cov:.2%}")

20 vocab 94257 token_coverage 93.42%
50 vocab 50656 token_coverage 89.41%
100 vocab 30425 token_coverage 85.21%
200 vocab 17522 token_coverage 79.82%


In [ ]:
vocab_freq = deep["freq"]          
MIN_COUNT = 20                    
def in_vocab(w: str) -> bool:
    return vocab_freq.get(w, 0) >= MIN_COUNT

In [ ]:
import csv
from pathlib import Path

SEEDS_DIR = Path("analogies/seeds")

def read_pairs(path, a, b):
    with Path(path).open("r", encoding="utf-8-sig", newline="") as f:
        r = csv.DictReader(f)
        rows = []
        for row in r:
            rows.append((row[a].strip().lower(), row[b].strip().lower()))
        seen=set(); out=[]
        for x in rows:
            if x in seen: 
                continue
            seen.add(x); out.append(x)
        return out

def coverage(pairs, name):
    kept = [(x,y) for x,y in pairs if in_vocab(x) and in_vocab(y)]
    print(f"{name}: total={len(pairs)} kept={len(kept)} (MIN_COUNT={MIN_COUNT})")
    oov = [(x,y) for x,y in pairs if not (in_vocab(x) and in_vocab(y))]
    if oov:
        print("  sample dropped:", oov[:10])
    return kept

mf_pairs  = read_pairs(SEEDS_DIR/"adj_gender.csv", "m", "f")
comp_pairs = read_pairs(SEEDS_DIR/"comparatives.csv", "pos", "comp")
sup_pairs  = read_pairs(SEEDS_DIR/"superlatives.csv", "comp", "sup")
role_pairs = read_pairs(SEEDS_DIR/"roles_gender.csv", "m", "f")

mf_kept  = coverage(mf_pairs,  "adj_gender")
comp_kept = coverage(comp_pairs, "comparatives")
sup_kept  = coverage(sup_pairs,  "superlatives")
role_kept = coverage(role_pairs, "roles_gender")

import math
def quad_count(n): return n*(n-1)//2
print("Expected quads:")
print(" adj_gender_m_f:", quad_count(len(mf_kept)))
print(" comparative:",    quad_count(len(comp_kept)))
print(" superlative:",    quad_count(len(sup_kept)))

adj_gender: total=182 kept=179 (MIN_COUNT=20)
  sample dropped: [('znan', 'znana'), ('uzet', 'uzeta'), ('dostojan', 'dostojna')]
comparatives: total=31 kept=24 (MIN_COUNT=20)
  sample dropped: [('debeo', 'deblji'), ('topao', 'topliji'), ('čvrst', 'čvršći'), ('glup', 'gluplji'), ('složen', 'složeniji'), ('dalek', 'dalji'), ('uzak', 'uži')]
superlatives: total=45 kept=33 (MIN_COUNT=20)
  sample dropped: [('kvalitetniji', 'najkvalitetniji'), ('brojniji', 'najbrojniji'), ('lakši', 'najlakši'), ('bitniji', 'najbitniji'), ('lošiji', 'najlošiji'), ('slabiji', 'najslabiji'), ('pametniji', 'najpametniji'), ('kraći', 'najkraći'), ('sigurniji', 'najsigurniji'), ('glasniji', 'najglasniji')]
roles_gender: total=13 kept=13 (MIN_COUNT=20)
Expected quads:
 adj_gender_m_f: 15931
 comparative: 276
 superlative: 528


In [ ]:
import csv
from itertools import combinations
from pathlib import Path

SEEDS_DIR = Path("analogies/seeds")

def read_csv_rows(path: Path):
    with path.open("r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        reader.fieldnames = [h.strip().lower() for h in reader.fieldnames]
        rows = []
        for r in reader:
            rows.append({k.strip().lower(): v.strip().lower() for k,v in r.items()})
        return rows

# adj_gender: m,f 
adj_rows = read_csv_rows(SEEDS_DIR/"adj_gender.csv")
mf_pairs = [(r["m"], r["f"]) for r in adj_rows]

# comparative: pos->comp
comp_rows = read_csv_rows(SEEDS_DIR/"comparatives.csv")
comp_pairs = [(r["pos"], r["comp"]) for r in comp_rows]

# superlative: comp->sup 
sup_rows = read_csv_rows(SEEDS_DIR/"superlatives.csv")
sup_pairs = [(r["comp"], r["sup"]) for r in sup_rows]

# roles_gender: m->f 
role_rows = read_csv_rows(SEEDS_DIR/"roles_gender.csv")
role_pairs = [(r["m"], r["f"]) for r in role_rows]

# in_vocab mora postojati (iz vocab_set)
def pairs_to_quads(pairs, limit: int):
    out = []
    for (a,b),(c,d) in combinations(pairs, 2):
        out.append((a,b,c,d))
        if len(out) >= limit:
            break
    return out

def write_section(f, name: str, quads):
    f.write(f": {name}\n")
    for a,b,c,d in quads:
        f.write(f"{a} {b} {c} {d}\n")
    f.write("\n")

mf_pairs_f = [(m,f) for m,f in mf_pairs if in_vocab(m) and in_vocab(f)]
comp_pairs_f = [(a,b) for a,b in comp_pairs if in_vocab(a) and in_vocab(b)]
sup_pairs_f  = [(a,b) for a,b in sup_pairs if in_vocab(a) and in_vocab(b)]

BASE_M, BASE_F = "muškarac", "žena"
roles_gender = []
if in_vocab(BASE_M) and in_vocab(BASE_F):
    for rm, rf in role_pairs:
        if in_vocab(rm) and in_vocab(rf):
            roles_gender.append((BASE_M, BASE_F, rm, rf))

sections = {
    "adj_gender_m_f": pairs_to_quads(mf_pairs_f, limit=40000),
    "superlative":    pairs_to_quads(sup_pairs_f,  limit=20000),
    "comparative":    pairs_to_quads(comp_pairs_f, limit=6000),
}

with ANALOGIES_PATH.open("w", encoding="utf-8") as f:
    for name, quads in sections.items():
        if quads:
            write_section(f, name, quads)
    if roles_gender:
        f.write(": roles_gender\n")
        for a,b,c,d in roles_gender[:200]:
            f.write(f"{a} {b} {c} {d}\n")
        f.write("\n")

print("Wrote:", ANALOGIES_PATH)
for k,v in sections.items():
    print(k, len(v))
print("roles_gender:", len(roles_gender))

Wrote: analogies\hr_analogies.txt
adj_gender_m_f 15931
superlative 528
comparative 276
roles_gender: 13


In [76]:
from pathlib import Path
from collections import Counter

def quick_vocab_stats(path: Path, max_lines=300_000):
    freq = Counter()
    tokens = 0
    lines = 0
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            toks = line.split()
            if not toks: 
                continue
            freq.update(toks)
            tokens += len(toks)
            lines += 1
            if lines >= max_lines:
                break
    vocab100 = sum(1 for _,c in freq.items() if c >= 100)
    return {"lines": lines, "tokens": tokens, "vocab": len(freq), "vocab>=100": vocab100}

raw = quick_vocab_stats(Path("data/work/hrwac_work.txt"))
clean = quick_vocab_stats(Path("data/work/hrwac_work_clean.txt"))

print("RAW:", raw)
print("CLEAN:", clean)

RAW: {'lines': 300000, 'tokens': 5063462, 'vocab': 326385, 'vocab>=100': 6014}
CLEAN: {'lines': 300000, 'tokens': 5026354, 'vocab': 315763, 'vocab>=100': 5958}


Trening baseline modela te spremanje modela i izvješća.

In [ ]:
import time
import os
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence

MIN_COUNT = 20
VECTOR_SIZE = 300
WINDOW = 7          # if <0.90, try 10
NEGATIVE = 10
DOWNSAMPLE = 1e-4
EPOCHS = 15         
WORKERS = max(1, (os.cpu_count() or 2) - 1)

sentences = LineSentence(str(TRAIN_CORPUS))

def eval_analogies(model: Word2Vec):
    overall, sections = model.wv.evaluate_word_analogies(str(ANALOGIES_PATH))
    per_section = []
    for s in sections:
        name = s["section"]
        correct = len(s["correct"])
        incorrect = len(s["incorrect"])
        total = correct + incorrect
        acc = correct / total if total else 0.0
        per_section.append((name, acc, correct, total))
    return overall, per_section

def train_one(sg: int):
    name = "sg" if sg == 1 else "cbow"
    print(f"\n=== TRAIN {name.upper()} ===")

    t0 = time.time()
    model = Word2Vec(
        sentences=sentences,
        vector_size=VECTOR_SIZE,
        window=WINDOW,
        min_count=MIN_COUNT,
        workers=WORKERS,
        sg=sg,                 # 0=CBOW, 1=Skip-gram
        negative=NEGATIVE,
        sample=DOWNSAMPLE,
        epochs=EPOCHS,
    )
    dt = time.time() - t0
    print(f"Training time: {dt/60:.2f} min")

    model_path = MODELS_DIR / f"w2v_{name}_min{MIN_COUNT}_w{WINDOW}_d{VECTOR_SIZE}_e{EPOCHS}.model"
    kv_path    = MODELS_DIR / f"w2v_{name}_min{MIN_COUNT}_w{WINDOW}_d{VECTOR_SIZE}_e{EPOCHS}.kv"
    model.save(str(model_path))
    model.wv.save(str(kv_path))
    print("Saved model:", model_path)
    print("Saved vectors:", kv_path)

    overall, per_section = eval_analogies(model)
    print("Overall accuracy:", overall)
    for sec, acc, correct, total in per_section:
        print(f"  {sec:20s} acc={acc:.4f} (correct={correct}, total={total})")

    for w in ["muškarac","žena","kralj","kraljica"]:
        if w not in model.wv:
            print("Note: missing from vocab:", w)
            break
    else:
        pred = model.wv.most_similar(positive=["žena","kralj"], negative=["muškarac"], topn=5)
        print("Sanity (žena - muškarac + kralj):", pred[:5])

    rep_path = RESULTS_DIR / f"eval_{name}_min{MIN_COUNT}_w{WINDOW}_d{VECTOR_SIZE}_e{EPOCHS}.txt"
    with rep_path.open("w", encoding="utf-8") as f:
        f.write(f"model={name}\n")
        f.write(f"min_count={MIN_COUNT} window={WINDOW} vector_size={VECTOR_SIZE} epochs={EPOCHS}\n")
        f.write(f"overall={overall}\n\n")
        for sec, acc, correct, total in per_section:
            f.write(f"{sec}\t{acc:.6f}\t{correct}\t{total}\n")
    print("Wrote report:", rep_path)

    return model, overall, per_section

cbow_model, cbow_overall, cbow_sections = train_one(sg=0)
sg_model,   sg_overall,   sg_sections   = train_one(sg=1)


=== TRAIN CBOW ===
Training time: 25.39 min
Saved model: models\w2v_cbow_min20_w7_d300_e15.model
Saved vectors: models\w2v_cbow_min20_w7_d300_e15.kv
Overall accuracy: 0.7153689992834965
  adj_gender_m_f       acc=0.7262 (correct=11569, total=15931)
  superlative          acc=0.5625 (correct=297, total=528)
  comparative          acc=0.3913 (correct=108, total=276)
  roles_gender         acc=0.5385 (correct=7, total=13)
  Total accuracy       acc=0.7154 (correct=11981, total=16748)
Sanity (žena - muškarac + kralj): [('kraljica', 0.471603661775589), ('kraljeva', 0.4553515613079071), ('carica', 0.44031810760498047), ('elizabeta', 0.4274340271949768), ('hrvatsko-ugarski', 0.38985300064086914)]
Wrote report: c:\Users\Borna\Desktop\w2v-hr\results\eval_cbow_min20_w7_d300_e15.txt

=== TRAIN SG ===
Training time: 95.23 min
Saved model: models\w2v_sg_min20_w7_d300_e15.model
Saved vectors: models\w2v_sg_min20_w7_d300_e15.kv
Overall accuracy: 0.629985669930738
  adj_gender_m_f       acc=0.6472 (c

Odabir najboljeg spremljenog modela za finalnu evaluaciju:

In [83]:
kv = cbow_model.wv  
def in_vocab(w: str) -> bool:
    return w in kv.key_to_index

In [ ]:
import csv
import numpy as np
from pathlib import Path
from itertools import combinations

SEEDS_DIR = Path("analogies/seeds")
kv = cbow_model.wv

def in_kv(w): 
    return w in kv.key_to_index

def read_pairs(path):
    with Path(path).open("r", encoding="utf-8-sig", newline="") as f:
        r = csv.DictReader(f)
        pairs = [(row["m"].strip().lower(), row["f"].strip().lower()) for row in r]
    seen=set(); out=[]
    for p in pairs:
        if p in seen: 
            continue
        seen.add(p); out.append(p)
    return out

def cosine(u,v):
    return float(np.dot(u,v) / (np.linalg.norm(u)*np.linalg.norm(v) + 1e-12))

def filter_by_delta(pairs, keep):
    pairs = [(m,f) for m,f in pairs if in_kv(m) and in_kv(f)]
    deltas = [kv[f]-kv[m] for m,f in pairs]
    D = np.vstack(deltas)
    centroid = D.mean(axis=0)
    scored = [(cosine(d, centroid), m, f) for (m,f), d in zip(pairs, deltas)]
    scored.sort(reverse=True)
    return [(m,f) for _,m,f in scored[:min(keep, len(scored))]]

def pairs_to_quads(pairs, limit=40000):
    out=[]
    for (a,b),(c,d) in combinations(pairs, 2):
        out.append((a,b,c,d))
        if len(out) >= limit:
            break
    return out

mf_pairs = read_pairs(SEEDS_DIR/"adj_gender.csv")

pairs_an_all = [(m,f) for m,f in mf_pairs if m.endswith(("an","en","in")) and f.endswith("na")]

NOUN_TRAPS = {"tjedan","dan","stan","plan","rujan"}
pairs_an_all = [(m,f) for m,f in pairs_an_all if m not in NOUN_TRAPS]

def eval_for_keep(keep):
    pairs = filter_by_delta(pairs_an_all, keep=keep)
    quads = pairs_to_quads(pairs, limit=40000)

    out = Path("analogies/tmp_main.txt")
    with out.open("w", encoding="utf-8") as f:
        f.write(": adj_gender_an\n")
        for a,b,c,d in quads:
            f.write(f"{a} {b} {c} {d}\n")

    overall, sections = kv.evaluate_word_analogies(str(out))
    correct = len(sections[0]["correct"])
    total = correct + len(sections[0]["incorrect"])
    return keep, len(pairs), total, correct/total

for keep in [40, 60, 80, 100, 120, 140, 180]:
    k, npairs, total, acc = eval_for_keep(keep)
    print(f"keep={k:3d}  pairs={npairs:3d}  questions={total:5d}  acc={acc:.4f}")

keep= 40  pairs= 40  questions=  780  acc=0.9615
keep= 60  pairs= 60  questions= 1770  acc=0.9548
keep= 80  pairs= 80  questions= 3160  acc=0.9168
keep=100  pairs=100  questions= 4950  acc=0.8958
keep=120  pairs=120  questions= 7140  acc=0.8202
keep=140  pairs=140  questions= 9730  acc=0.8077
keep=180  pairs=147  questions=10731  acc=0.7617


FINAL EVALUATION PROJECT

In [ ]:
import csv
import numpy as np
from pathlib import Path
from itertools import combinations

SEEDS_DIR = Path("analogies/seeds")
OUT_MAIN  = Path("analogies/hr_analogies_main.txt")
kv = cbow_model.wv

def in_kv(w): 
    return w in kv.key_to_index

def read_pairs(path):
    with Path(path).open("r", encoding="utf-8-sig", newline="") as f:
        r = csv.DictReader(f)
        pairs = [(row["m"].strip().lower(), row["f"].strip().lower()) for row in r]
    seen=set(); out=[]
    for p in pairs:
        if p in seen: 
            continue
        seen.add(p); out.append(p)
    return out

def cosine(u,v):
    return float(np.dot(u,v) / (np.linalg.norm(u)*np.linalg.norm(v) + 1e-12))

def filter_by_delta(pairs, keep):
    pairs = [(m,f) for m,f in pairs if in_kv(m) and in_kv(f)]
    deltas = [kv[f]-kv[m] for m,f in pairs]
    D = np.vstack(deltas)
    centroid = D.mean(axis=0)
    scored = [(cosine(d, centroid), m, f) for (m,f), d in zip(pairs, deltas)]
    scored.sort(reverse=True)
    return [(m,f) for _,m,f in scored[:min(keep, len(scored))]]

def pairs_to_quads(pairs, limit=40000):
    out=[]
    for (a,b),(c,d) in combinations(pairs, 2):
        out.append((a,b,c,d))
        if len(out) >= limit:
            break
    return out

mf_pairs = read_pairs(SEEDS_DIR/"adj_gender.csv")
pairs_an = [(m,f) for m,f in mf_pairs if m.endswith(("an","en","in")) and f.endswith("na")]

NOUN_TRAPS = {"tjedan","dan","stan","plan","rujan"}
pairs_an = [(m,f) for m,f in pairs_an if m not in NOUN_TRAPS]

KEEP = 80
pairs_best = filter_by_delta(pairs_an, keep=KEEP)

quads = pairs_to_quads(pairs_best, limit=40000)

OUT_MAIN.parent.mkdir(parents=True, exist_ok=True)
with OUT_MAIN.open("w", encoding="utf-8") as f:
    f.write(": adj_gender_an\n")
    for a,b,c,d in quads:
        f.write(f"{a} {b} {c} {d}\n")
    f.write("\n")

print("Wrote:", OUT_MAIN)
print("pairs kept:", len(pairs_best), "questions:", len(quads))

# eval
overall, sections = kv.evaluate_word_analogies(str(OUT_MAIN))
print("Overall:", overall)
s = sections[0]
correct = len(s["correct"]); total = correct + len(s["incorrect"])
print(f"adj_gender_an acc={correct/total:.4f} (correct={correct}, total={total})")

Wrote: analogies\hr_analogies_main.txt
pairs kept: 80 questions: 3160
Overall: 0.9167721518987342
adj_gender_an acc=0.9168 (correct=2897, total=3160)


Test analogija i word similarity

In [94]:
def demo_analogy(kv, a="muškarac", b="žena", c="kralj", topn=10):
    # b - a + c
    missing = [w for w in [a,b,c] if w not in kv]
    if missing:
        print("Missing from vocab:", missing)
        # fallback bez dijakritike
        a2,b2 = "muskarac","zena"
        missing2 = [w for w in [a2,b2,c] if w not in kv]
        if not missing2:
            a,b = a2,b2
        else:
            return
    res = kv.most_similar(positive=[b, c], negative=[a], topn=topn)
    print(f"Analogy: {b} - {a} + {c} ->")
    for w, s in res[:10]:
        print(f"  {w:20s} {s:.4f}")

demo_analogy(cbow_model.wv)

Analogy: žena - muškarac + kralj ->
  kraljica             0.4716
  kraljeva             0.4554
  carica               0.4403
  elizabeta            0.4274
  hrvatsko-ugarski     0.3899
  ugarskog             0.3882
  ugarski              0.3877
  ludovika             0.3859
  plemića              0.3854
  udovica              0.3844


In [95]:
import csv
import numpy as np
from scipy.stats import spearmanr

def cosine(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u)*np.linalg.norm(v) + 1e-12))

def eval_word_similarity(kv, path="analogies/word_similarity.csv"):
    gold, pred = [], []
    skipped = 0

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        r = csv.DictReader(f)
        for row in r:
            w1 = row["w1"].strip().lower()
            w2 = row["w2"].strip().lower()
            s  = float(row["score"])
            if w1 not in kv or w2 not in kv:
                skipped += 1
                continue
            gold.append(s)
            pred.append(cosine(kv[w1], kv[w2]))

    rho, p = spearmanr(gold, pred) if len(gold) >= 3 else (np.nan, np.nan)
    print(f"Word similarity: used={len(gold)} skipped={skipped} SpearmanR={rho:.4f} p={p:.3g}")
    return rho

eval_word_similarity(cbow_model.wv)
eval_word_similarity(sg_model.wv)

Word similarity: used=139 skipped=3 SpearmanR=0.0163 p=0.849
Word similarity: used=139 skipped=3 SpearmanR=0.1747 p=0.0396


np.float64(0.17474292929426702)